# The KISS: Conditional NCA Training and Evaluation

Use this notebook to explore the deep-learning side of the project before choosing the final `webapp/model_config.yaml` settings. It supports 64x64 Klimt targets, 3x3 or 5x5 neighborhoods, growth/persistence/transition objectives, metrics, plots, checkpointing, and GIF previews.

In [ ]:
from pathlib import Path
import sys

sys.path.append('src')

import matplotlib.pyplot as plt
import pandas as pd
import torch

from kiss_nca.config import resolve_device
from kiss_nca.dataset import load_paintings
from kiss_nca.model import ConditionalNCA
from kiss_nca.training import (
    clamp_rgb,
    rollout_frames,
    save_checkpoint,
    save_gif,
    train_model,
    transition_frames,
)

## Experiment configuration

Change these values to compare `64x64, 3x3` versus `64x64, 5x5`, hidden channels, condition dimension, training length, batch size, and device.

In [ ]:
CONFIG = {
    'model_name': 'kiss_nca_experiment',
    'resolution': 64,
    'neighborhood_size': 3,  # choose 3 or 5
    'state_channels': 16,
    'hidden_channels': 16,
    'condition_dim': 16,
    'update_rate': 0.5,
    'dataset_dir': 'paintings/64',
    'paintings': ['the_kiss', 'adele_bloch_bauer', 'tree_of_life', 'judith', 'danae'],
    'training': {
        'iterations': 500,
        'batch_size': 8,
        'steps_min': 48,
        'steps_max': 96,
        'learning_rate': 1e-3,
        'lambda_persist': 0.5,
        'lambda_transition': 0.25,
        'use_growth': True,
        'use_persistence': True,
        'use_transition': True,
    },
    'device': 'auto',
    'output': {
        'model_path': 'experiments/checkpoints/kiss_nca_experiment.pt',
        'metadata_path': 'experiments/checkpoints/metadata.json',
    },
}

device = resolve_device(CONFIG['device'])
device

## Load and validate painting targets

In [ ]:
targets, records = load_paintings(CONFIG['dataset_dir'], CONFIG['paintings'], device=device)
print(targets.shape)
print([r.name for r in records])

fig, axes = plt.subplots(1, len(records), figsize=(12, 3))
for ax, record, target in zip(axes, records, targets.cpu()):
    ax.imshow(target.permute(1, 2, 0))
    ax.set_title(record.name)
    ax.axis('off')
plt.show()

## Build and train one conditional NCA

In [ ]:
model = ConditionalNCA(
    state_channels=CONFIG['state_channels'],
    hidden_channels=CONFIG['hidden_channels'],
    num_paintings=len(records),
    condition_dim=CONFIG['condition_dim'],
    neighborhood_size=CONFIG['neighborhood_size'],
    update_rate=CONFIG['update_rate'],
).to(device)

history = train_model(model, targets, CONFIG['training'])
history_df = pd.DataFrame(history)
history_df.tail()

## Training loss curves and metrics table

In [ ]:
history_df.set_index('iteration')[['total', 'growth', 'persistence', 'transition']].plot(figsize=(10, 4), logy=True)
plt.ylabel('loss')
plt.show()

metrics = history_df[['total', 'growth', 'persistence', 'transition']].tail(50).mean().to_frame('last_50_mean')
metrics

## Current generated preview versus target

In [ ]:
condition_id = 0
frames = rollout_frames(model, condition_id=condition_id, resolution=64, total_steps=CONFIG['training']['steps_max'], every=CONFIG['training']['steps_max'], device=device)
preview = frames[-1]

fig, axes = plt.subplots(1, 2, figsize=(5, 3))
axes[0].imshow(preview.permute(1, 2, 0))
axes[0].set_title('Generated')
axes[0].axis('off')
axes[1].imshow(targets[condition_id].detach().cpu().permute(1, 2, 0))
axes[1].set_title(f'Target: {records[condition_id].name}')
axes[1].axis('off')
plt.show()

## Save checkpoint and GIF previews

In [ ]:
save_checkpoint(model, CONFIG, records, CONFIG['output']['model_path'], CONFIG['output']['metadata_path'])
print('Checkpoint:', CONFIG['output']['model_path'])
print('Metadata:', CONFIG['output']['metadata_path'])

growth_path = Path('experiments/gifs') / f"{records[0].name}_64_{CONFIG['neighborhood_size']}x{CONFIG['neighborhood_size']}_growth.gif"
growth_frames = rollout_frames(model, condition_id=0, resolution=64, total_steps=96, every=4, device=device)
save_gif(growth_frames, growth_path)
print('Growth GIF:', growth_path)

transition_path = Path('experiments/gifs') / f"{records[0].name}_to_{records[2].name}_64_{CONFIG['neighborhood_size']}x{CONFIG['neighborhood_size']}_transition.gif"
transition_preview = transition_frames(model, source_id=0, target_id=2, resolution=64, grow_steps=64, transition_steps=64, every=4, device=device)
save_gif(transition_preview, transition_path)
print('Transition GIF:', transition_path)

## Export final configuration manually

After comparing experiments, copy the best hyperparameters into `webapp/model_config.yaml`. The future web app will load exactly one model trained by `python train_script.py --config webapp/model_config.yaml`.